# 🏆 Ensemble & Stacking - Final Submission Pipeline

- **Thành viên:** C (Phạm Hồng Nam) & Nhóm
- **Mã thí nghiệm:** `ens_c_v1`
- **Kết quả:** CV AUC = **0.94732**, Public LB = **0.94760** (Top score của nhóm!)
- **Phương pháp tối ưu:** **Rank Averaging** kết hợp 3 mô hình đa dạng (LightGBM v2 + XGBoost v1 + CatBoost v1).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "C" else Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "C" / "outputs"
SUB_DIR = ROOT / "submissions"

train_df = pd.read_csv(DATA_DIR / "train.csv", usecols=["id", "is_fraud"])
sub_sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
y_true = train_df["is_fraud"].values

models = {
    "LightGBM v2": {
        "oof": np.load(ROOT / "A/outputs/lgbm_a_v2_oof.npy"),
        "test": np.load(ROOT / "A/outputs/lgbm_a_v2_test.npy")
    },
    "XGBoost v1": {
        "oof": np.load(ROOT / "B/outputs/xgb_b_v1_oof.npy"),
        "test": np.load(ROOT / "B/outputs/xgb_b_v1_test.npy")
    },
    "CatBoost v1": {
        "oof": np.load(ROOT / "C/outputs/cat_c_v1_oof.npy"),
        "test": np.load(ROOT / "C/outputs/cat_c_v1_test.npy")
    }
}

for name, d in models.items():
    print(f"{name:<15}: OOF AUC = {roc_auc_score(y_true, d['oof']):.5f}")


In [ ]:
# Tương quan dự đoán
corr_df = pd.DataFrame({k: v["oof"] for k, v in models.items()}).corr()
print("Ma trận tương quan Pearson giữa OOF các mô hình:")
print(corr_df.round(4))


In [ ]:
# 1. Simple Average
auc_avg = roc_auc_score(y_true, (models["LightGBM v2"]["oof"] + models["XGBoost v1"]["oof"] + models["CatBoost v1"]["oof"]) / 3.0)

# 2. Rank Average
oof_rank = (rankdata(models["LightGBM v2"]["oof"]) + rankdata(models["XGBoost v1"]["oof"]) + rankdata(models["CatBoost v1"]["oof"])) / (3.0 * len(y_true))
test_rank = (rankdata(models["LightGBM v2"]["test"]) + rankdata(models["XGBoost v1"]["test"]) + rankdata(models["CatBoost v1"]["test"])) / (3.0 * len(sub_sample))
auc_rank = roc_auc_score(y_true, oof_rank)

# 3. Weighted Blend
oof_mat = np.column_stack([d["oof"] for d in models.values()])
test_mat = np.column_stack([d["test"] for d in models.values()])
def opt_fn(w):
    w = w / np.sum(w)
    return -roc_auc_score(y_true, np.dot(oof_mat, w))
res = minimize(opt_fn, [1/3, 1/3, 1/3], method="SLSQP", bounds=[(0, 1)]*3)
w_opt = res.x / np.sum(res.x)
auc_blend = roc_auc_score(y_true, np.dot(oof_mat, w_opt))

print(f"Simple Average OOF AUC: {auc_avg:.5f}")
print(f"Rank Average OOF AUC:   {auc_rank:.5f}")
print(f"Weighted Blend OOF AUC: {auc_blend:.5f} (Weights: {w_opt.round(3)})")


In [ ]:
# Xuất file nộp chính
np.save(OUTPUT_DIR / "ens_c_v1_oof.npy", oof_rank)
np.save(OUTPUT_DIR / "ens_c_v1_test.npy", test_rank)

sub_sample["is_fraud"] = test_rank
sub_sample.to_csv(SUB_DIR / "ens_c_v1.csv", index=False)
print("Xuất file nộp submissions/ens_c_v1.csv thành công!")
